# 学习 Pi05 LoRA 微调：从 checkpoint 到 adapter

这个 Notebook 只围绕 VLA 的 Pi05，帮助你分清：

- 从头训练；
- 从 `pi05_libero_base` 全量微调；
- 从已有 checkpoint 全量继续微调；
- 从已有 checkpoint 进行 LoRA 微调。

默认不会加载 7 GB 以上模型，也不会启动 `lerobot-train`。先读懂参数，再手动打开开关。

## 目标

理解这一条实验链路：

```text
已有 Pi05 checkpoint
  -> policy.path 加载 config + weights
  -> --peft.method_type=LORA 包装 policy
  -> 冻结基础权重，训练 adapter/指定 modules_to_save
  -> 保存 LoRA checkpoint
  -> 用同一个 simulator benchmark 评测
```

注意：LoRA 不是“把所有模型参数都训练一遍”。Pi05 当前默认 LoRA target 主要是 action expert 的部分 attention 投影和 action/state projection。

## 设置

训练依赖建议在复现环境中安装：

```bash
UV_PROJECT_ENVIRONMENT=.venv-libero uv sync --locked \
  --extra training --extra evaluation --extra libero --extra pi --extra peft
```

先保持两个开关为 `False`。`RUN_MODEL_INSPECTION=True` 会加载模型并统计参数；`RUN_TRAIN=True` 才会真的启动训练。

In [ ]:
import shlex
import subprocess

# 安全开关：第一次运行 Notebook 时都保持 False。
RUN_MODEL_INSPECTION = False
RUN_TRAIN = False

DATASET_REPO = "lerobot/libero"
BASE_CHECKPOINT = "lerobot/pi05_libero_base"
FINETUNED_CHECKPOINT = "lerobot/pi05_libero_finetuned_v044"
OUTPUT_DIR = "outputs/reproduction/pi05_lora"
STEPS = 10000
SEED = 1000

print(f"DATASET_REPO={DATASET_REPO}")
print(f"BASE_CHECKPOINT={BASE_CHECKPOINT}")
print(f"FINETUNED_CHECKPOINT={FINETUNED_CHECKPOINT}")
print(f"RUN_MODEL_INSPECTION={RUN_MODEL_INSPECTION}, RUN_TRAIN={RUN_TRAIN}")

## 1. 四种训练方式先分清

| 方式 | 典型参数 | 含义 |
| --- | --- | --- |
| 从头训练 | `--policy.type=pi05`，没有 checkpoint | 模型随机初始化，LIBERO 小数据通常不适合 |
| 基础模型全量微调 | `--policy.pretrained_path=lerobot/pi05_libero_base` | 加载权重，默认更新全部可训练参数 |
| 已微调模型继续全量微调 | `--policy.path=lerobot/pi05_libero_finetuned_v044` | 从已有强 checkpoint 继续更新全部参数 |
| LoRA 微调 | `--policy.path=...` 加 `--peft.method_type=LORA` | 基础权重冻结，主要更新低秩 adapter |

## 2. 先构造全量微调命令

这个命令来自 LeRobot 的 Pi05 文档。这里不执行，只把命令拆成 Python list，便于检查参数。两个 freeze 开关都为 false 时，表示全量微调。

In [ ]:
full_finetune_command = [
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO}",
    "--policy.type=pi05",
    f"--policy.pretrained_path={BASE_CHECKPOINT}",
    '--policy.normalization_mapping={"ACTION":"MEAN_STD","STATE":"MEAN_STD","VISUAL":"IDENTITY"}',
    "--policy.n_action_steps=10",
    "--policy.empty_cameras=1",
    "--policy.freeze_vision_encoder=false",
    "--policy.train_expert_only=false",
    "--policy.dtype=bfloat16",
    "--policy.device=cuda",
    "--policy.push_to_hub=false",
    "--output_dir=outputs/reproduction/pi05_full",
    "--steps=30000",
    "--batch_size=64",
    f"--seed={SEED}",
]

print(shlex.join(full_finetune_command))

## 3. 再构造 LoRA 命令

这里从已经在 LIBERO 上微调过的 `pi05_libero_finetuned_v044` 开始。这个实验应被称为“二阶段 LoRA 适配”，不要和从 `pi05_libero_base` 复现官方训练混为一谈。

In [ ]:
lora_command = [
    "lerobot-train",
    f"--dataset.repo_id={DATASET_REPO}",
    f"--policy.path={FINETUNED_CHECKPOINT}",
    "--peft.method_type=LORA",
    "--peft.r=16",
    "--peft.lora_alpha=32",
    f"--output_dir={OUTPUT_DIR}",
    "--job_name=pi05_lora",
    "--policy.push_to_hub=false",
    "--batch_size=32",
    f"--steps={STEPS}",
    "--save_freq=2000",
    f"--seed={SEED}",
]

print(shlex.join(lora_command))
print("\n这只是打印命令，RUN_TRAIN 仍然是 False。")

## 4. 可选：加载模型并统计参数

这一步会真正下载和加载 Pi05 权重，可能占用大量显存。只有在已经完成 LIBERO 冒烟评测、确认显存足够后，才把 `RUN_MODEL_INSPECTION` 改为 `True`。

In [ ]:
if RUN_MODEL_INSPECTION:
    import torch
    from lerobot.policies.pi05 import PI05Policy

    policy = PI05Policy.from_pretrained(FINETUNED_CHECKPOINT)
    total_parameters = sum(parameter.numel() for parameter in policy.parameters())
    trainable_parameters = sum(
        parameter.numel() for parameter in policy.parameters() if parameter.requires_grad
    )
    print(f"total parameters: {total_parameters:,}")
    print(f"trainable before PEFT: {trainable_parameters:,}")
    print("注意：这里还没有调用 wrap_with_peft，因此显示的是基础模型的可训练参数。")
else:
    print("已跳过模型加载：确认显存后把 RUN_MODEL_INSPECTION 改为 True。")

## 5. 训练开关

训练时建议不要在 Notebook 内反复点击执行。确认命令、数据集、输出目录和 GPU 后，在终端执行相同命令更容易保存日志。这个 Cell 只是给出受控的 Notebook 入口。

In [ ]:
if RUN_TRAIN:
    # LoRA 训练前应确保环境包含 --extra peft。
    # 另外，Pi05 的 EMA shadow 不应和 PEFT adapter 一起使用。
    subprocess.run(lora_command, check=True)
else:
    print("RUN_TRAIN=False：没有启动训练。请复制上一个 Cell 打印的命令到终端执行。")

## 检查

开始真正训练前，逐项确认：

- `lerobot/pi05_libero_finetuned_v044` 可以单独通过 `lerobot-eval`；
- LoRA 输出目录没有和输入 checkpoint 相同；
- `--peft.method_type=LORA` 已安装 PEFT extra；
- 没有打开 `--ema.enable=true`；
- 数据集、seed、steps、batch size 和 baseline 保持可比较；
- 训练后使用相同的 LIBERO/LIBERO-plus 脚本评测。

## 下一步

第一轮不要马上比较 LoRA 和全量微调的最终论文结论。先完成：

1. `pi05_libero_finetuned_v044` 的 LIBERO 冒烟；
2. 同一 checkpoint 的 LIBERO-plus 冒烟；
3. 再决定 LoRA 是从 `pi05_libero_base` 还是从已微调 checkpoint 开始；
4. 记录 trainable parameter 数量、显存、训练步数和成功率。